# Payments: more training passes, tested after each

Continues training the model you trained earlier for one more pass over the same 800 examples, then tests it alone and with the rules. This gives a second point on the accuracy-by-pass curve. About 3 hours per pass.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**, and upload `adapter_model.safetensors` and `adapter_config.json` when asked.
3. Keep the tab open. `payments_pass2.zip` downloads at the end; send it to Claude.

To run more than one extra pass, change `MORE_PASSES` in the last cell before running.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti

In [ ]:
import os, sys, json, time, glob, zipfile
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()   # log times in Indian time, shutil
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=4096, load_in_4bit=True, full_finetuning=False)

In [ ]:
# Upload the two files you saved from the first payments run: adapter_model.safetensors and adapter_config.json
from google.colab import files
os.makedirs("/content/pay_adapter", exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, "/content/pay_adapter/" + ("adapter_model.safetensors" if name.endswith(".safetensors") else "adapter_config.json"))
print(sorted(os.listdir("/content/pay_adapter")))
assert {"adapter_model.safetensors", "adapter_config.json"} <= set(os.listdir("/content/pay_adapter")), "Upload both files."

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/pay_adapter", is_trainable=True)

import stack
def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stack.HF_CHAT = hf_chat

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
tools_text = json.dumps(json.load(open("tools.json")), indent=1)
template = open("prompts/baseline.txt").read()
def prompt_of(t):
    return template.replace("{tools}", tools_text).replace("{account}", json.dumps(t["account"], indent=1)).replace("{request}", t["request"])
rows = []
for t in json.load(open("train_tasks.json"))[:800]:          # the same 800 examples as the first pass
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt_of(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": json.dumps(t["answer"])}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
ds = Dataset.from_list(rows)
print(len(rows), "training examples")

In [ ]:
MORE_PASSES = 1          # each extra pass takes about 2.5 hours, plus about 40 minutes of testing
os.makedirs("results", exist_ok=True)
for p in range(2, 2 + MORE_PASSES):
    model.train()
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=5,
        num_train_epochs=1, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407 + p, report_to="none", output_dir="outputs"))
    trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
    t0 = time.time(); trainer.train()
    json.dump({"pass": p, "minutes": round((time.time() - t0) / 60, 1), "log": trainer.state.log_history}, open("results/finetune_log_pass%d.json" % p, "w"))
    model.save_pretrained("/content/pay_adapter_pass%d" % p)
    model.eval()
    for layers in ["0", "1234"]:
        stack.main(["hf:gemma-3-4b-tuned", "--tasks", "tasks_third.json", "--layers", layers, "--tag", "ep%d" % p, "--workers", "1", "--budget", "100000"])
    with zipfile.ZipFile("/content/payments_pass%d.zip" % p, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob("results/*ep%d.jsonl" % p) + ["results/finetune_log_pass%d.json" % p]:
            z.write(f)
    from google.colab import files
    files.download("/content/payments_pass%d.zip" % p)